In [1]:
from getpass import getuser # Libaray to copy things
from pathlib import Path # Object oriented libary to deal with paths
from dask.utils import format_bytes
from distributed import Client, progress, wait # Libaray to orchestrate distributed resources
#from dask_jobqueue import SLURMCluster # Setting up distributed memories via slurm
import numpy as np # Pythons standard array library
import xarray as xr # Libary to work with labeled n-dimensional data
import dask # Distributed data libary
import dask.distributed
import multiprocessing
from dask.diagnostics import ProgressBar
import glob, os
from matplotlib.lines import Line2D

from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter1d
from subprocess import run, PIPE
import sys
import os
from sklearn.neighbors import KernelDensity
import gc
from concurrent.futures import ProcessPoolExecutor
import seaborn as sns
import warnings
warnings.filterwarnings(action='ignore')
dask.config.config.get('distributed').get('dashboard').update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})
import pandas as pd
from scipy.interpolate import CloughTocher2DInterpolator, LinearNDInterpolator, NearestNDInterpolator

In [2]:
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter, AutoMinorLocator)
from matplotlib import pyplot as plt # Standard Plotting library
from matplotlib import cm
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter, AutoMinorLocator)
from cartopy import crs as ccrs # Cartography library
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
import matplotlib.tri as tri

In [3]:
#'''Provides information about the CPU and starts the dask client'''
ncpu = multiprocessing.cpu_count()
processes = False
nworker = 1
threads = ncpu // nworker
print(f"Number of CPUs: {ncpu}, number of threads: {threads}, number of workers: {nworker}, processes: {processes}")
client = Client(processes=processes,
 threads_per_worker=threads,
 n_workers=nworker,
 memory_limit='1000GB'
 )

Number of CPUs: 64, number of threads: 64, number of workers: 1, processes: False


In [4]:
latitude = [30,60]


In [5]:
ifs_on = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/ifs_2.8km_on_30-60N_precip_total_sum.nc')['tp']
ifs_off = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/ifs_2.8km_off_30-60N_precip_total_sum.nc')['tp']
ifs_cycle3 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/ifs_cycle3_30-60N_precip_total_sum.nc')['tp']
ifs_cycle4 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/ifs_cycle4_1990-2020_30-60N_precip_total_sum.nc')['tp']
icon_cycle3 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/icon_cycle3_30-60N_precip_total_sum.nc')['pr']
icon_cycle4 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/icon_cycle4_2021-2049_30-60N_precip_total_sum.nc')['pr']
# CESM2-LE: 10 hourly members, mean May-Oct total 1850-2014, ensemble mean (cesm2_hourly_precip_total_cdo.sh)
cesm2 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/mid_lat_precip/cesm2_ens_hourly_total/cesm2_hourly_precip_total_ensmean_1850-2014_30-60N.nc')['PRECT'].squeeze(drop=True)
imerg = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/imerg_30-60N_precip_total_sum.nc')['precipitation']
#ifs_cycle4 = ifs_cycle4.isel(year=slice(0,30))
#ifs_cycle4 = ifs_cycle4.stack(sample=("year", "time"))
icon_cycle4 = icon_cycle4.isel(year=slice(0,5))
#icon_cycle4 = icon_cycle4.stack(sample=("year", "time"))



In [6]:
land_sea_ifs_high = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_2.8km_deep_off/ifs_hpx2048_lsm_30-60N.nc')['lsm']
land_sea_ifs_cycle3 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_4.4-FESOM_5/IFS_cycle3_lsm_30-60N.nc')['lsm']
land_sea_ifs_zoom9 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_zoom9/30-60N_ifs_hpx512_lsm.nc')['lsm']
land_sea_icon_zoom10 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/NextGEMS/ICON_zoom10/icon_cycle3_lsm_30-60N.nc')['ocean_fraction_surface']
land_sea_icon_zoom9 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/NextGEMS/ICON_zoom9/30-60N_ICON_ngc4008_ocean_fraction_surface_zoom9.nc')['ocean_fraction_surface']
land_sea_imerg = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/IMERG/clean/30-60N_fixgrid_GPM_IMERG_LandSeaMask.2.nc4')['landseamask']
land_sea_imerg = land_sea_imerg.assign_coords(lat=imerg.lat, lon=imerg.lon)
land_sea_cesm2 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/CESM2/precip/24H/mask.nc')['MASK']
land_sea_cesm2 = land_sea_cesm2[0,:,:]
indexer = ((land_sea_cesm2.lat>=latitude[0]) & (land_sea_cesm2.lat<=latitude[1]))
land_sea_cesm2 = land_sea_cesm2.where(indexer,drop=True)
land_sea_cesm2 = land_sea_cesm2.assign_coords(lat=cesm2.lat, lon=cesm2.lon)

ifs_on_land = ifs_on.where(land_sea_ifs_high > 0, np.nan)
ifs_off_land = ifs_off.where(land_sea_ifs_high > 0, np.nan)
ifs_cycle3_land = ifs_cycle3.where(land_sea_ifs_cycle3 > 0, np.nan)
ifs_cycle4_land = ifs_cycle4.where(land_sea_ifs_zoom9 > 0, np.nan)
icon_cycle3_land = icon_cycle3.where(land_sea_icon_zoom10 < 1, np.nan)
icon_cycle4_land = icon_cycle4.where(land_sea_icon_zoom9 < 1, np.nan)
imerg_land = imerg.where(land_sea_imerg < 100, np.nan)
cesm2_land = cesm2.where(land_sea_cesm2 > 0, np.nan)

ifs_on_sea = ifs_on.where(land_sea_ifs_high == 0, np.nan)
ifs_off_sea = ifs_off.where(land_sea_ifs_high == 0, np.nan)
ifs_cycle3_sea = ifs_cycle3.where(land_sea_ifs_cycle3 == 0, np.nan)
ifs_cycle4_sea = ifs_cycle4.where(land_sea_ifs_zoom9 == 0, np.nan)
icon_cycle3_sea = icon_cycle3.where(land_sea_icon_zoom10 == 1, np.nan)
icon_cycle4_sea = icon_cycle4.where(land_sea_icon_zoom9 == 1, np.nan)
imerg_sea = imerg.where(land_sea_imerg == 100, np.nan)
cesm2_sea = cesm2.where(land_sea_cesm2 == 0, np.nan)



In [7]:
print((land_sea_ifs_high > 0).sum())

<xarray.DataArray 'lsm' ()> Size: 8B
array(4697561)


In [8]:
print(icon_cycle4_land.mean() / 1000)

<xarray.DataArray 'pr' ()> Size: 8B
array(0.45738591)


In [9]:
ifs_on_land_mean = ifs_on_land.mean() / 1000
ifs_off_land_mean = ifs_off_land.mean() / 1000
ifs_cycle3_land_mean = ifs_cycle3_land.mean() / 1000
ifs_cycle4_land_mean = ifs_cycle4_land.mean() / 1000
icon_cycle3_land_mean = icon_cycle3_land.mean() / 1000
icon_cycle4_land_mean = icon_cycle4_land.mean() / 1000
# IMERG and CESM2 are on regular lat-lon grids: weight by cos(lat) (cell area). IFS/ICON grids are
# (nearly) equal-area, so their plain mean is already an area mean.
w_imerg = np.cos(np.deg2rad(imerg_land.lat))
w_cesm2 = np.cos(np.deg2rad(cesm2_land.lat))
imerg_land_mean = imerg_land.weighted(w_imerg).mean(('lat', 'lon')) / 1000
cesm2_land_mean = cesm2_land.weighted(w_cesm2).mean(('lat', 'lon')) / 1000

ifs_on_sea_mean = ifs_on_sea.mean() / 1000
ifs_off_sea_mean = ifs_off_sea.mean() / 1000
ifs_cycle3_sea_mean = ifs_cycle3_sea.mean() / 1000
ifs_cycle4_sea_mean = ifs_cycle4_sea.mean() / 1000
icon_cycle3_sea_mean = icon_cycle3_sea.mean() / 1000
icon_cycle4_sea_mean = icon_cycle4_sea.mean() / 1000
imerg_sea_mean = imerg_sea.weighted(w_imerg).mean(('lat', 'lon')) / 1000
cesm2_sea_mean = cesm2_sea.weighted(w_cesm2).mean(('lat', 'lon')) / 1000

In [10]:
print('land area mean precip for ifs 2.8 on is ' + str(ifs_on_land_mean.values) + 'm')
print('land area mean precip for ifs 2.8 off is ' + str(ifs_off_land_mean.values) + 'm')
print('land area mean precip for ifs cycle3 is ' + str(ifs_cycle3_land_mean.values) + 'm')
print('land area mean precip for ifs cycle4 is ' + str(ifs_cycle4_land_mean.values) + 'm')
print('land area mean precip for icon cycle3 is ' + str(icon_cycle3_land_mean.values) + 'm')
print('land area mean precip for icon cycle4 is ' + str(icon_cycle4_land_mean.values) + 'm')

print('land area mean precip for imerg is ' + str(imerg_land_mean.values) + 'm')
print('land area mean precip for cesm2 is ' + str(cesm2_land_mean.values) + 'm')


land area mean precip for ifs 2.8 on is 0.35393079350648554m
land area mean precip for ifs 2.8 off is 0.37443290949521174m
land area mean precip for ifs cycle3 is 0.3557400558708479m
land area mean precip for ifs cycle4 is 0.3429350319555415m
land area mean precip for icon cycle3 is 0.3746657409667969m
land area mean precip for icon cycle4 is 0.4573859118941723m
land area mean precip for imerg is 0.36328543090820314m
land area mean precip for cesm2 is 0.35871286373785005m


In [11]:
print('sea area mean precip for ifs 2.8 on is ' + str(ifs_on_sea_mean.values) + 'm')
print('sea area mean precip for ifs 2.8 off is ' + str(ifs_off_sea_mean.values) + 'm')
print('sea area mean precip for ifs cycle3 is ' + str(ifs_cycle3_sea_mean.values) + 'm')
print('sea area mean precip for ifs cycle4 is ' + str(ifs_cycle4_sea_mean.values) + 'm')
print('sea area mean precip for icon cycle3 is ' + str(icon_cycle3_sea_mean.values) + 'm')
print('sea area mean precip for icon cycle4 is ' + str(icon_cycle4_sea_mean.values) + 'm')

print('sea area mean precip for imerg is ' + str(imerg_sea_mean.values) + 'm')
print('sea area mean precip for cesm2 is ' + str(cesm2_sea_mean.values) + 'm')


sea area mean precip for ifs 2.8 on is 0.5307222290313772m
sea area mean precip for ifs 2.8 off is 0.5391348633539189m
sea area mean precip for ifs cycle3 is 0.5215825794830509m
sea area mean precip for ifs cycle4 is 0.528124539747796m
sea area mean precip for icon cycle3 is 0.5589088745117188m
sea area mean precip for icon cycle4 is 0.5319523540178047m
sea area mean precip for imerg is 0.56991357421875m
sea area mean precip for cesm2 is 0.4980505795369704m
